# Single-chain context

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

## Test runs

The next cells will run the model training for different pooling layer strategies.

This process can take a long time to finish.

**If you already have the models trained, you can skip this section and go to the evaluation section.**

In [ ]:
dataset_path = OUTPUTS_ROOT.absolute() / "_dataset"
fixed_args = {
    "device": "cpu",  # Change to "cuda" if you have a GPU and want to use it
    "dataset": DATASET_TYPE.SINGLE,
    "load_data": str(dataset_path),
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,
    "tags": "deepsad,tuningpooling",

    "num_epochs": 100,
    "learning_rate": 0.005,
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "residual": False,
    "random_seed": 42,
    "test_split": 0.15,
    "rep_dim": 64,
    "eta": 1.0,
    "deep_sad_eps": 1.0,
    "grad_clip": 1.0,
    "weight_decay": 1e-6,
    "ae_pretrain": False,
}

In [ ]:
# Run for mean pooling
pooling = "mean"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for max pooling
pooling = "max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for sum pooling
pooling = "sum"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for mean+max pooling
pooling = "mean_max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

## Results processing

In [3]:
# Obtain all relevant run directories that match the pattern and contain a params.yaml file
run_dirs = sorted(
    p for p in OUTPUTS_ROOT.glob("train_epochs*")
    if p.is_dir() and (p / "params.yaml").exists()
)

print(f"Found {len(run_dirs)} run directories")
for p in run_dirs:
    print(p)

Found 4 run directories
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_deepsadbased__202608051947
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_poolmax_nosf_es15_deepsadbased_tuningpooling__202608051954
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_poolmean_max_nosf_es15_deepsadbased_tuningpooling__202608052008
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augme

In [4]:
import yaml
import pandas as pd

def load_run_config(run_dir: Path) -> dict:
    TOTAL_METAPATHS = 6475  # Total number of metapaths in the dataset
    with open(run_dir / "params.yaml") as f:
        doc = yaml.safe_load(f)

    return {
        "run_dir": run_dir,
        "run_name": run_dir.name,
        "pooling": doc["params"]["pooling"],
    }

configs = [load_run_config(p) for p in run_dirs]
configs_df = pd.DataFrame(configs)
configs_df

,run_dir,run_name,pooling
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_nosf_es15_dee...,mean
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmax_nosf_...,max
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmean_max_...,mean_max
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolsum_nosf_...,sum


In [5]:
def load_val_metrics(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "val_best_metrics.csv")
    metric_cols = [c for c in df.columns if c != "fold"]

    result = {"run_dir": run_dir}
    for col in metric_cols:
        result[f"{col}_mean"] = df[col].mean()
        result[f"{col}_std"] = df[col].std(ddof=0)
    
    result["precision_macro_mean"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).mean()
    result["precision_macro_std"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).std(ddof=0)
    result["recall_macro_mean"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).mean()
    result["recall_macro_std"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).std(ddof=0)
    result["f1_macro_mean"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).mean()
    result["f1_macro_std"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).std(ddof=0)
    return result

metrics = [load_val_metrics(p) for p in run_dirs]
metrics_df = pd.DataFrame(metrics)
metrics_df

,run_dir,epoch_mean,epoch_std,val_loss_mean,val_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,precision_normal_std,recall_normal_mean,...,mcc_mean,mcc_std,threshold_mean,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,15.4,6.590903,1.913616,3.578951,0.956070,0.013565,0.996008,0.003261,0.948881,...,0.879162,0.033631,0.140854,0.066711,0.914140,0.026051,0.966877,0.007794,0.936395,0.018677
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,18.6,13.544002,0.815784,1.363275,0.928493,0.043181,0.990629,0.011816,0.919353,...,0.819736,0.083581,0.146275,0.111413,0.881006,0.054084,0.942113,0.031640,0.902021,0.050575
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,9.8,3.544009,1.185886,2.012686,0.927504,0.034927,0.987261,0.015602,0.922081,...,0.815144,0.062729,0.265278,0.298210,0.883063,0.052370,0.935720,0.019682,0.899355,0.040262
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,16.0,12.915107,0.453572,0.501240,0.893715,0.035646,0.980148,0.010140,0.884885,...,0.732521,0.074032,0.110162,0.070826,0.830113,0.044360,0.906994,0.029842,0.855944,0.042794


In [6]:
def load_fold_durations(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "fold_durations.csv")
    return {
        "run_dir": run_dir,
        "fold_time_mean": df["time"].mean(),
        "fold_time_std": df["time"].std(ddof=0),  # match summary.txt convention
        "fold_time_total": df["time"].sum(),
    }

durations = [load_fold_durations(p) for p in run_dirs]
durations_df = pd.DataFrame(durations)
durations_df


,run_dir,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,77.896,17.181736,389.48
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,82.650,32.678488,413.25
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,63.146,9.293074,315.73
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,70.034,33.622929,350.17


In [7]:
comparison_df = (
    configs_df
    .merge(metrics_df, on="run_dir")
    .merge(durations_df, on="run_dir")
)
comparison_df["pooling"] = pd.Categorical(comparison_df["pooling"], categories=["mean", "max", "sum", "mean_max"])
comparison_df = (
    comparison_df
    .sort_values(["pooling"])
    .reset_index(drop=True)
)
comparison_df

,run_dir,run_name,pooling,epoch_mean,epoch_std,val_loss_mean,val_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,...,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_nosf_es15_dee...,mean,15.4,6.590903,1.913616,3.578951,0.956070,0.013565,0.996008,...,0.066711,0.914140,0.026051,0.966877,0.007794,0.936395,0.018677,77.896,17.181736,389.48
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmax_nosf_...,max,18.6,13.544002,0.815784,1.363275,0.928493,0.043181,0.990629,...,0.111413,0.881006,0.054084,0.942113,0.031640,0.902021,0.050575,82.650,32.678488,413.25
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolsum_nosf_...,sum,16.0,12.915107,0.453572,0.501240,0.893715,0.035646,0.980148,...,0.070826,0.830113,0.044360,0.906994,0.029842,0.855944,0.042794,70.034,33.622929,350.17
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmean_max_...,mean_max,9.8,3.544009,1.185886,2.012686,0.927504,0.034927,0.987261,...,0.298210,0.883063,0.052370,0.935720,0.019682,0.899355,0.040262,63.146,9.293074,315.73


In [8]:
trimmed_df = comparison_df[[
    "pooling",
    "accuracy_mean",
    "accuracy_std",
    "precision_macro_mean",
    "precision_macro_std",
    "recall_macro_mean",
    "recall_macro_std",
    "f1_macro_mean",
    "f1_macro_std",
]]
trimmed_df

,pooling,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,mean,0.956070,0.013565,0.914140,0.026051,0.966877,0.007794,0.936395,0.018677
1,max,0.928493,0.043181,0.881006,0.054084,0.942113,0.031640,0.902021,0.050575
2,sum,0.893715,0.035646,0.830113,0.044360,0.906994,0.029842,0.855944,0.042794
3,mean_max,0.927504,0.034927,0.883063,0.052370,0.935720,0.019682,0.899355,0.040262


In [9]:
PCT_SCALE_EXCLUDE = {"fold_time"}  # keep seconds as seconds, not *100

def format_stat(mean, std, base):
    scale = 1 if base in PCT_SCALE_EXCLUDE else 100
    return f"{mean * scale:.2f} \u00b1 {std * scale:05.2f}"

metric_bases = [c[:-5] for c in trimmed_df.columns if c.endswith("_mean")]

display_df2 = pd.DataFrame(index=trimmed_df.index)
for base in metric_bases:
    display_df2[base] = trimmed_df.apply(
        lambda row: format_stat(row[f"{base}_mean"], row[f"{base}_std"], base), axis=1
    )

display_df2.insert(0, "pooling", trimmed_df["pooling"])
display_df2 = display_df2.set_index("pooling")

final_table = display_df2
final_table


,accuracy,precision_macro,recall_macro,f1_macro
pooling,,,,
mean,95.61 ± 01.36,91.41 ± 02.61,96.69 ± 00.78,93.64 ± 01.87
max,92.85 ± 04.32,88.10 ± 05.41,94.21 ± 03.16,90.20 ± 05.06
sum,89.37 ± 03.56,83.01 ± 04.44,90.70 ± 02.98,85.59 ± 04.28
mean_max,92.75 ± 03.49,88.31 ± 05.24,93.57 ± 01.97,89.94 ± 04.03


In [10]:
# Turn the unicode ± into LaTeX math syntax, and escape underscores in row labels
import re

METRIC_LABELS = {
    "pooling": "Pooling Type",
    "accuracy": "Accuracy (\\%)",
    "precision_macro": "Precision (\\%)",
    "recall_macro": "Recall (\\%)",
    "f1_macro": "F1-Score (\\%)",
}

POOLING_LABELS = {
    "mean": "Mean Pooling",
    "max": "Max Pooling",
    "sum": "Sum Pooling",
    "mean_max": "Mean + Max Pooling",
}

def to_math_pm(s: str) -> str:
    return re.sub(r"(-?[\d.]+) ± (-?[\d.]+)", r"\1 $\\pm$ \2", s)

latex_ready = final_table.map(to_math_pm)
latex_ready = latex_ready.reset_index()  # turn the "pooling" index back into a plain column
latex_ready = latex_ready.rename(columns=METRIC_LABELS)
latex_ready["Pooling Type"] = latex_ready["Pooling Type"].map(POOLING_LABELS)

latex_str = latex_ready.to_latex(
    index=False,  # drop the numeric row index; "Pooling Type" is now a real column
    escape=False,  # cell values already contain LaTeX ($\pm$), don't double-escape
    column_format="l" + "r" * (len(latex_ready.columns) - 1),
    caption="Effect of the DME threshold on test-set performance",
    label="tab:ablation_semantic_fusion",
)
print(latex_str)


\begin{table}
\caption{Effect of the DME threshold on test-set performance}
\label{tab:ablation_semantic_fusion}
\begin{tabular}{lrrrr}
\toprule
Pooling Type & Accuracy (\%) & Precision (\%) & Recall (\%) & F1-Score (\%) \\
\midrule
Mean Pooling & 95.61 $\pm$ 01.36 & 91.41 $\pm$ 02.61 & 96.69 $\pm$ 00.78 & 93.64 $\pm$ 01.87 \\
Max Pooling & 92.85 $\pm$ 04.32 & 88.10 $\pm$ 05.41 & 94.21 $\pm$ 03.16 & 90.20 $\pm$ 05.06 \\
Sum Pooling & 89.37 $\pm$ 03.56 & 83.01 $\pm$ 04.44 & 90.70 $\pm$ 02.98 & 85.59 $\pm$ 04.28 \\
Mean + Max Pooling & 92.75 $\pm$ 03.49 & 88.31 $\pm$ 05.24 & 93.57 $\pm$ 01.97 & 89.94 $\pm$ 04.03 \\
\bottomrule
\end{tabular}
\end{table}



# Cross-chain context

In [11]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

## Test runs

The next cells will run the model training for different pooling layer strategies.

This process can take a long time to finish.

**If you already have the models trained, you can skip this section and go to the evaluation section.**

In [29]:
dataset_path = OUTPUTS_ROOT.absolute() / "_dataset"
fixed_args = {
    "device": "cpu",  # Change to "cuda" if you have a GPU and want to use it
    "dataset": DATASET_TYPE.CCTX,
    "load_data": str(dataset_path),
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,
    "tags": "deepsad,tuningpooling",

    "num_epochs": 100,
    "learning_rate": 0.002, # Notice that the new model uses a lower learning rate compared to the single-chain context
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "residual": False,
    "random_seed": 42,
    "test_split": 0.15,
    "rep_dim": 64,
    "eta": 1.0,
    "deep_sad_eps": 1.0,
    "grad_clip": 1.0,
    "weight_decay": 1e-6,
    "ae_pretrain": False,
}

In [ ]:
# Run for mean pooling
pooling = "mean"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for max pooling
pooling = "max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for sum pooling
pooling = "sum"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for mean+max pooling
pooling = "mean_max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

## Results processing

In [12]:
# Obtain all relevant run directories that match the pattern and contain a params.yaml file
run_dirs = sorted(
    p for p in OUTPUTS_ROOT.glob("train_epochs*")
    if p.is_dir() and (p / "params.yaml").exists()
)

print(f"Found {len(run_dirs)} run directories")
for p in run_dirs:
    print(p)

Found 4 run directories
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_deepsadbased__202608061807
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_poolmax_nosf_es15_deepsadbased_tuningpooling__202608062018
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_poolmean_max_nosf_es15_deepsadbased_tuningpooling__202608062103
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.

In [13]:
import yaml
import pandas as pd

def load_run_config(run_dir: Path) -> dict:
    TOTAL_METAPATHS = 6475  # Total number of metapaths in the dataset
    with open(run_dir / "params.yaml") as f:
        doc = yaml.safe_load(f)

    return {
        "run_dir": run_dir,
        "run_name": run_dir.name,
        "pooling": doc["params"]["pooling"],
    }

configs = [load_run_config(p) for p in run_dirs]
configs_df = pd.DataFrame(configs)
configs_df

,run_dir,run_name,pooling
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_nosf_es15_dee...,mean
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmax_nosf_...,max
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmean_max_...,mean_max
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolsum_nosf_...,sum


In [14]:
def load_val_metrics(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "val_best_metrics.csv")
    metric_cols = [c for c in df.columns if c != "fold"]

    result = {"run_dir": run_dir}
    for col in metric_cols:
        result[f"{col}_mean"] = df[col].mean()
        result[f"{col}_std"] = df[col].std(ddof=0)
    
    result["precision_macro_mean"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).mean()
    result["precision_macro_std"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).std(ddof=0)
    result["recall_macro_mean"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).mean()
    result["recall_macro_std"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).std(ddof=0)
    result["f1_macro_mean"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).mean()
    result["f1_macro_std"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).std(ddof=0)
    return result

metrics = [load_val_metrics(p) for p in run_dirs]
metrics_df = pd.DataFrame(metrics)
metrics_df

,run_dir,epoch_mean,epoch_std,val_loss_mean,val_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,precision_normal_std,recall_normal_mean,...,mcc_mean,mcc_std,threshold_mean,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,20.0,7.042727,0.053216,0.063440,0.997152,0.002325,0.996959,0.002483,1.000000,...,0.977200,0.018638,15.696655,22.749816,0.998480,0.001241,0.979048,0.017130,0.988382,0.009497
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,20.6,19.126944,0.215867,0.257352,0.971559,0.042704,0.995810,0.002111,0.973485,...,0.865586,0.157839,63.236588,124.859788,0.916100,0.117119,0.958647,0.028125,0.924617,0.094230
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,18.8,5.306600,0.093694,0.095929,0.990512,0.012359,0.995897,0.003866,0.993878,...,0.931694,0.083899,0.451239,0.439691,0.963028,0.055484,0.969320,0.029466,0.965378,0.042663
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,30.8,9.744742,0.139709,0.147634,0.993342,0.009329,0.997949,0.002513,0.994903,...,0.950804,0.067042,1.181734,1.840650,0.968302,0.046683,0.983166,0.020760,0.974935,0.034361


In [15]:
def load_fold_durations(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "fold_durations.csv")
    return {
        "run_dir": run_dir,
        "fold_time_mean": df["time"].mean(),
        "fold_time_std": df["time"].std(ddof=0),  # match summary.txt convention
        "fold_time_total": df["time"].sum(),
    }

durations = [load_fold_durations(p) for p in run_dirs]
durations_df = pd.DataFrame(durations)
durations_df


,run_dir,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,185.444,32.417810,927.22
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,223.528,106.015011,1117.64
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,214.960,39.186509,1074.80
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,281.856,58.711656,1409.28


In [16]:
comparison_df = (
    configs_df
    .merge(metrics_df, on="run_dir")
    .merge(durations_df, on="run_dir")
)
comparison_df["pooling"] = pd.Categorical(comparison_df["pooling"], categories=["mean", "max", "sum", "mean_max"])
comparison_df = (
    comparison_df
    .sort_values(["pooling"])
    .reset_index(drop=True)
)
comparison_df

,run_dir,run_name,pooling,epoch_mean,epoch_std,val_loss_mean,val_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,...,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_nosf_es15_dee...,mean,20.0,7.042727,0.053216,0.063440,0.997152,0.002325,0.996959,...,22.749816,0.998480,0.001241,0.979048,0.017130,0.988382,0.009497,185.444,32.417810,927.22
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmax_nosf_...,max,20.6,19.126944,0.215867,0.257352,0.971559,0.042704,0.995810,...,124.859788,0.916100,0.117119,0.958647,0.028125,0.924617,0.094230,223.528,106.015011,1117.64
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolsum_nosf_...,sum,30.8,9.744742,0.139709,0.147634,0.993342,0.009329,0.997949,...,1.840650,0.968302,0.046683,0.983166,0.020760,0.974935,0.034361,281.856,58.711656,1409.28
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmean_max_...,mean_max,18.8,5.306600,0.093694,0.095929,0.990512,0.012359,0.995897,...,0.439691,0.963028,0.055484,0.969320,0.029466,0.965378,0.042663,214.960,39.186509,1074.80


In [17]:
trimmed_df = comparison_df[[
    "pooling",
    "accuracy_mean",
    "accuracy_std",
    "precision_macro_mean",
    "precision_macro_std",
    "recall_macro_mean",
    "recall_macro_std",
    "f1_macro_mean",
    "f1_macro_std",
]]
trimmed_df

,pooling,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,mean,0.997152,0.002325,0.998480,0.001241,0.979048,0.017130,0.988382,0.009497
1,max,0.971559,0.042704,0.916100,0.117119,0.958647,0.028125,0.924617,0.094230
2,sum,0.993342,0.009329,0.968302,0.046683,0.983166,0.020760,0.974935,0.034361
3,mean_max,0.990512,0.012359,0.963028,0.055484,0.969320,0.029466,0.965378,0.042663


In [18]:
PCT_SCALE_EXCLUDE = {"fold_time"}  # keep seconds as seconds, not *100

def format_stat(mean, std, base):
    scale = 1 if base in PCT_SCALE_EXCLUDE else 100
    return f"{mean * scale:.2f} \u00b1 {std * scale:05.2f}"

metric_bases = [c[:-5] for c in trimmed_df.columns if c.endswith("_mean")]

display_df2 = pd.DataFrame(index=trimmed_df.index)
for base in metric_bases:
    display_df2[base] = trimmed_df.apply(
        lambda row: format_stat(row[f"{base}_mean"], row[f"{base}_std"], base), axis=1
    )

display_df2.insert(0, "pooling", trimmed_df["pooling"])
display_df2 = display_df2.set_index("pooling")

final_table2 = display_df2
final_table2


,accuracy,precision_macro,recall_macro,f1_macro
pooling,,,,
mean,99.72 ± 00.23,99.85 ± 00.12,97.90 ± 01.71,98.84 ± 00.95
max,97.16 ± 04.27,91.61 ± 11.71,95.86 ± 02.81,92.46 ± 09.42
sum,99.33 ± 00.93,96.83 ± 04.67,98.32 ± 02.08,97.49 ± 03.44
mean_max,99.05 ± 01.24,96.30 ± 05.55,96.93 ± 02.95,96.54 ± 04.27


In [19]:
# Turn the unicode ± into LaTeX math syntax, and escape underscores in row labels
import re

METRIC_LABELS = {
    "pooling": "Pooling Type",
    "accuracy": "Accuracy (\\%)",
    "precision_macro": "Precision (\\%)",
    "recall_macro": "Recall (\\%)",
    "f1_macro": "F1-Score (\\%)",
}

POOLING_LABELS = {
    "mean": "Mean Pooling",
    "max": "Max Pooling",
    "sum": "Sum Pooling",
    "mean_max": "Mean + Max Pooling",
}

def to_math_pm(s: str) -> str:
    return re.sub(r"(-?[\d.]+) ± (-?[\d.]+)", r"\1 $\\pm$ \2", s)

latex_ready2 = final_table2.map(to_math_pm)
latex_ready2 = latex_ready2.reset_index()  # turn the "pooling" index back into a plain column
latex_ready2 = latex_ready2.rename(columns=METRIC_LABELS)
latex_ready2["Pooling Type"] = latex_ready2["Pooling Type"].map(POOLING_LABELS)

latex_str = latex_ready2.to_latex(
    index=False,  # drop the numeric row index; "Pooling Type" is now a real column
    escape=False,  # cell values already contain LaTeX ($\pm$), don't double-escape
    column_format="l" + "r" * (len(latex_ready2.columns) - 1),
    caption="Effect of the DME threshold on test-set performance",
    label="tab:ablation_semantic_fusion",
)
print(latex_str)


\begin{table}
\caption{Effect of the DME threshold on test-set performance}
\label{tab:ablation_semantic_fusion}
\begin{tabular}{lrrrr}
\toprule
Pooling Type & Accuracy (\%) & Precision (\%) & Recall (\%) & F1-Score (\%) \\
\midrule
Mean Pooling & 99.72 $\pm$ 00.23 & 99.85 $\pm$ 00.12 & 97.90 $\pm$ 01.71 & 98.84 $\pm$ 00.95 \\
Max Pooling & 97.16 $\pm$ 04.27 & 91.61 $\pm$ 11.71 & 95.86 $\pm$ 02.81 & 92.46 $\pm$ 09.42 \\
Sum Pooling & 99.33 $\pm$ 00.93 & 96.83 $\pm$ 04.67 & 98.32 $\pm$ 02.08 & 97.49 $\pm$ 03.44 \\
Mean + Max Pooling & 99.05 $\pm$ 01.24 & 96.30 $\pm$ 05.55 & 96.93 $\pm$ 02.95 & 96.54 $\pm$ 04.27 \\
\bottomrule
\end{tabular}
\end{table}



# Putting it all together

In [20]:
# Combine the single-chain and cross-chain pooling comparison tables into one,
# tagging each block of rows with the context (model) it came from.

def prep_model_table(df: pd.DataFrame, model_label: str) -> pd.DataFrame:
    prepped = df.map(to_math_pm)
    prepped = prepped.reset_index()  # turn the "pooling" index back into a plain column
    prepped = prepped.rename(columns=METRIC_LABELS)
    prepped["Pooling Type"] = prepped["Pooling Type"].map(POOLING_LABELS)
    prepped.insert(0, "Model", model_label)
    return prepped

combined_df = pd.concat([
    prep_model_table(final_table, "DeepSAD-based (Single-Chain)"),
    prep_model_table(final_table2, "DeepSAD-based (Cross-Chain)"),
], ignore_index=True)
combined_df = combined_df.set_index(["Model", "Pooling Type"])
combined_df


Accuracy (\%)  \
Model                        Pooling Type                            
DeepSAD-based (Single-Chain) Mean Pooling        95.61 $\pm$ 01.36   
                             Max Pooling         92.85 $\pm$ 04.32   
                             Sum Pooling         89.37 $\pm$ 03.56   
                             Mean + Max Pooling  92.75 $\pm$ 03.49   
DeepSAD-based (Cross-Chain)  Mean Pooling        99.72 $\pm$ 00.23   
                             Max Pooling         97.16 $\pm$ 04.27   
                             Sum Pooling         99.33 $\pm$ 00.93   
                             Mean + Max Pooling  99.05 $\pm$ 01.24   

                                                    Precision (\%)  \
Model                        Pooling Type                            
DeepSAD-based (Single-Chain) Mean Pooling        91.41 $\pm$ 02.61   
                             Max Pooling         88.10 $\pm$ 05.41   
                             Sum Pooling         83.01 $\pm$ 04.44   
                             Mean + Max Pooling  88.31 $\pm$ 05.24   
DeepSAD-based (Cross-Chain)  Mean Pooling        99.85 $\pm$ 00.12   
                             Max Pooling         91.61 $\pm$ 11.71   
                             Sum Pooling         96.83 $\pm$ 04.67   
                             Mean + Max Pooling  96.30 $\pm$ 05.55   

                                                       Recall (\%)  \
Model                        Pooling Type                            
DeepSAD-based (Single-Chain) Mean Pooling        96.69 $\pm$ 00.78   
                             Max Pooling         94.21 $\pm$ 03.16   
                             Sum Pooling         90.70 $\pm$ 02.98   
                             Mean + Max Pooling  93.57 $\pm$ 01.97   
DeepSAD-based (Cross-Chain)  Mean Pooling        97.90 $\pm$ 01.71   
                             Max Pooling         95.86 $\pm$ 02.81   
                             Sum Pooling         98.32 $\pm$ 02.08   
                             Mean + Max Pooling  96.93 $\pm$ 02.95   

                                                     F1-Score (\%)  
Model                        Pooling Type                           
DeepSAD-based (Single-Chain) Mean Pooling        93.64 $\pm$ 01.87  
                             Max Pooling         90.20 $\pm$ 05.06  
                             Sum Pooling         85.59 $\pm$ 04.28  
                             Mean + Max Pooling  89.94 $\pm$ 04.03  
DeepSAD-based (Cross-Chain)  Mean Pooling        98.84 $\pm$ 00.95  
                             Max Pooling         92.46 $\pm$ 09.42  
                             Sum Pooling         97.49 $\pm$ 03.44  
                             Mean + Max Pooling  96.54 $\pm$ 04.27

In [21]:
# Build the LaTeX table by hand: a single header row (Model + Pooling Type + metrics),
# vertically-centered \multirow cells for the repeated "Model" values, and \midrule
# (instead of \cline) between the two model blocks.

flat_df = combined_df.reset_index()
metric_cols = [c for c in flat_df.columns if c not in ("Model", "Pooling Type")]

header = " & ".join(["Model", "Pooling Type", *metric_cols]) + r" \\"

body_lines = []
models = flat_df["Model"].unique()
for i, model in enumerate(models):
    group = flat_df[flat_df["Model"] == model]
    for j, (_, row) in enumerate(group.iterrows()):
        model_cell = f"\\multirow{{{len(group)}}}{{*}}{{{model}}}" if j == 0 else ""
        cells = [model_cell, row["Pooling Type"], *[str(v) for v in row[metric_cols]]]
        body_lines.append(" & ".join(cells) + r" \\")
    if i < len(models) - 1:
        body_lines.append(r"\midrule")

column_format = "ll" + "c" * len(metric_cols)
body = "\n".join(body_lines)

latex_str = (
    "\\begin{table}\n"
    "\\centering\n"
    "\\resizebox{\\linewidth}{!}{%\n"
    f"\\begin{{tabular}}{{{column_format}}}\n"
    "\\toprule\n"
    f"{header}\n"
    "\\midrule\n"
    f"{body}\n"
    "\\bottomrule\n"
    "\\end{tabular}}%\n"
    "\\caption{Effect of pooling strategy on test-set performance for both models, in both single-chain and cross-chain contexts}\n"
    "\\label{tab:pooling_comparison_combined}\n"
    "\\end{table}\n"
)
print(latex_str)


\begin{table}
\centering
\resizebox{\linewidth}{!}{%
\begin{tabular}{llcccc}
\toprule
Model & Pooling Type & Accuracy (\%) & Precision (\%) & Recall (\%) & F1-Score (\%) \\
\midrule
\multirow{4}{*}{DeepSAD-based (Single-Chain)} & Mean Pooling & 95.61 $\pm$ 01.36 & 91.41 $\pm$ 02.61 & 96.69 $\pm$ 00.78 & 93.64 $\pm$ 01.87 \\
 & Max Pooling & 92.85 $\pm$ 04.32 & 88.10 $\pm$ 05.41 & 94.21 $\pm$ 03.16 & 90.20 $\pm$ 05.06 \\
 & Sum Pooling & 89.37 $\pm$ 03.56 & 83.01 $\pm$ 04.44 & 90.70 $\pm$ 02.98 & 85.59 $\pm$ 04.28 \\
 & Mean + Max Pooling & 92.75 $\pm$ 03.49 & 88.31 $\pm$ 05.24 & 93.57 $\pm$ 01.97 & 89.94 $\pm$ 04.03 \\
\midrule
\multirow{4}{*}{DeepSAD-based (Cross-Chain)} & Mean Pooling & 99.72 $\pm$ 00.23 & 99.85 $\pm$ 00.12 & 97.90 $\pm$ 01.71 & 98.84 $\pm$ 00.95 \\
 & Max Pooling & 97.16 $\pm$ 04.27 & 91.61 $\pm$ 11.71 & 95.86 $\pm$ 02.81 & 92.46 $\pm$ 09.42 \\
 & Sum Pooling & 99.33 $\pm$ 00.93 & 96.83 $\pm$ 04.67 & 98.32 $\pm$ 02.08 & 97.49 $\pm$ 03.44 \\
 & Mean + Max Pooling & 9